In [ ]:
import os
import cv2
from tqdm import tqdm
import scipy.io as sio
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

In [68]:
BASE_DIR = "/mnt/e/ML/Stanford Cars/Dataset/"


def load_dataset_dataframe(base_dir, anno_file_name):
    devkit_dir = os.path.join(base_dir, "car_devkit/devkit")

    # loading the metadata
    meta = sio.loadmat(os.path.join(devkit_dir, "cars_meta.mat"))
    class_names = [c[0] for c in meta["class_names"][0]]

    # loading the training annotations
    annos = sio.loadmat(os.path.join(devkit_dir, anno_file_name))
    annotations = annos["annotations"][0]

    data = []
    for anno in annotations:
        fname = anno["fname"][0]

        bx1 = int(anno["bbox_x1"][0][0])
        bx2 = int(anno["bbox_x2"][0][0])
        by1 = int(anno["bbox_y1"][0][0])
        by2 = int(anno["bbox_y2"][0][0])


        if "class" in anno.dtype.names:
            class_id = anno["class"][0][0] - 1
            class_name = class_names[class_id]
        else:
            class_name = None

        data.append({"filename": fname, "class_name": class_name, "bbox_x1": bx1, "bbox_y1": by1, "bbox_x2": bx2, "bbox_y2": by2})

    return pd.DataFrame(data)

In [69]:



df_train_full = load_dataset_dataframe(
    base_dir=BASE_DIR, anno_file_name="cars_train_annos.mat"
)
df_test = load_dataset_dataframe(
    base_dir=BASE_DIR, anno_file_name="cars_test_annos.mat"
)


In [82]:
df_test.head()


,filename,class_name,bbox_x1,bbox_y1,bbox_x2,bbox_y2
0,00001.jpg,None,30,52,246,147
1,00002.jpg,None,100,19,576,203
2,00003.jpg,None,51,105,968,659
3,00004.jpg,None,67,84,581,407
4,00005.jpg,None,140,151,593,339


In [71]:
# def crop_and_save_locally(df, source_folder_name, output_folder_name):
#     # Source path on E: drive
#     src_dir = os.path.join(BASE_DIR, source_folder_name)
#     # Fast native Linux destination path 
#     dest_dir = os.path.join(BASE_DIR, output_folder_name)
#     os.makedirs(dest_dir, exist_ok=True)

#     print(f"Processing {source_folder_name}...")
#     for idx, row in tqdm(df.iterrows(), total=len(df)):
#         img_path = os.path.join(src_dir, row['filename'])
#         img = cv2.imread(img_path)
#         if img is None: 
#             continue
            
#         # Extract your newly populated coordinates
#         x1, y1, x2, y2 = row['bbox_x1'], row['bbox_y1'], row['bbox_x2'], row['bbox_y2']
        
#         # Crop to the car and resize immediately to 300x300
#         cropped = img[y1:y2, x1:x2]
        
#         # Safe fallback check in case bounding box values are clipped or corrupted
#         if cropped.size == 0:
#             resized = cv2.resize(img, (300, 300))
#         else:
#             resized = cv2.resize(cropped, (300, 300))
            
#         # Write directly to native Linux SSD
#         cv2.imwrite(os.path.join(dest_dir, row['filename']), resized)

# # Run the processing sequence for both split folders
# # (Assuming your image subfolders are named 'cars_train' and 'cars_test')
# crop_and_save_locally(df_train_full, "cars_train", "train")
# crop_and_save_locally(df_test, "cars_test", "test")

Processing cars_train...


100%|██████████| 8144/8144 [11:26<00:00, 11.86it/s]


Processing cars_test...


100%|██████████| 8041/8041 [10:17<00:00, 13.02it/s]


In [72]:
# selected_classes = df_train_full["class_name"].unique()[:40]

# df_train_full = df_train_full[
#     df_train_full["class_name"].isin(selected_classes)
# ]

# df_test = df_test[df_test["class_name"].isin(selected_classes)]


In [73]:
df_train, df_val = train_test_split(
    df_train_full, test_size=0.2, random_state=42, stratify=df_train_full["class_name"]
)

print(
    f"Train samples: {len(df_train)} | Val samples: {len(df_val)} | Test samples: {len(df_test)}"
)

Train samples: 6515 | Val samples: 1629 | Test samples: 8041


In [74]:

TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train")
TEST_IMG_DIR = os.path.join(BASE_DIR, "test")

IMG_SIZE = (300, 300)
BATCH_SIZE = 8

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
)
# validation/Test sets remains clean
val_test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    directory=TRAIN_IMG_DIR,
    x_col="filename",
    y_col="class_name",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val,
    directory=TRAIN_IMG_DIR,
    x_col="filename",
    y_col="class_name",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_test,
    directory=TEST_IMG_DIR,
    x_col="filename",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False,
)

print("\n Pipeline Ready!")


Found 6515 validated image filenames belonging to 196 classes.
Found 1629 validated image filenames belonging to 196 classes.
Found 8041 validated image filenames.

 Pipeline Ready!


In [75]:

def build_efficientnet_model(num_classes):
    # loading EfficientNetB0
    base_model = tf.keras.applications.EfficientNetB3(
        # top=False means output layer removed, we will add our own output layer
        include_top=False,
        weights="imagenet",
        input_shape=(300, 300, 3),
    )

    base_model.trainable = False

    phase1_model = models.Sequential(
        [
            base_model,
            layers.BatchNormalization(),  # added batch normalization layer to stabilize training and improve convergence
            layers.GlobalAveragePooling2D(),  # converts the 3d image to flat 1d vector so that it can be fed to the dense layer
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        ]
    )

    phase1_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return phase1_model


In [76]:
num_classes = len(train_generator.class_indices)

model = build_efficientnet_model(num_classes)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint(
        filepath="best_efficientnet.keras", monitor="val_loss", save_best_only=True
    ),
]

history = model.fit(
    train_generator, validation_data=val_generator, epochs=15, callbacks=callbacks, steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)

print("Training phase 1 complete! Starting fine-tuning...")

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
Epoch 1/15


I0000 00:00:1779359520.890455    7791 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_297976__.409
E0000 00:00:1779359526.875444    7791 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1779359527.283912    7791 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


257/815 ━━━━━━━━━━━━━━━━━━━━ 2:30 270ms/step - accuracy: 0.0370 - loss: 5.1256

I0000 00:00:1779359619.104171    7788 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_297976__.409
E0000 00:00:1779359626.440192    7788 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


815/815 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - accuracy: 0.0846 - loss: 4.6397

E0000 00:00:1779359856.780553    7790 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1779359861.771348    7790 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


815/815 ━━━━━━━━━━━━━━━━━━━━ 366s 406ms/step - accuracy: 0.1477 - loss: 4.0766 - val_accuracy: 0.3806 - val_loss: 2.7220
Epoch 2/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 281s 344ms/step - accuracy: 0.3793 - loss: 2.7199 - val_accuracy: 0.5328 - val_loss: 2.0468
Epoch 3/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 295s 361ms/step - accuracy: 0.4973 - loss: 2.1897 - val_accuracy: 0.5961 - val_loss: 1.7599
Epoch 4/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 297s 364ms/step - accuracy: 0.5613 - loss: 1.8737 - val_accuracy: 0.6231 - val_loss: 1.5803
Epoch 5/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 276s 338ms/step - accuracy: 0.6000 - loss: 1.7092 - val_accuracy: 0.6611 - val_loss: 1.4739
Epoch 6/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 272s 334ms/step - accuracy: 0.6407 - loss: 1.5673 - val_accuracy: 0.6722 - val_loss: 1.4346
Epoch 7/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 271s 332ms/step - accuracy: 0.6543 - loss: 1.5082 - val_accuracy: 0.7060 - val_loss: 1.3892
Epoch 8/15
815/815 ━━━━━━━━━━━━━━━━━━━━ 272s 334ms/step - accuracy: 0.6812 - loss: 1.41

In [77]:
def apply_fine_tuning(trained_model):

    base_model = trained_model.layers[0]

    base_model.trainable = True

    # Freeze the BatchNormalization layers
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    fine_tune = len(base_model.layers) - 40
    for layer in base_model.layers[:fine_tune]:  # selects from index 0 to index n-40
        layer.trainable = False

    trained_model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=3e-6, weight_decay=0.01),
        # instead of categorical crossentropy which gave 71% uisng label_smoothing
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
        metrics=["accuracy"],
    )

    return trained_model


In [78]:
best_phase1_model = tf.keras.models.load_model("best_efficientnet.keras")

model_finetuned = apply_fine_tuning(best_phase1_model)

callbacks_finetune = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint(
        filepath="best_efficientnet_finetuned.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=4,
        verbose=1,
        min_lr=1e-7,
    )
]
history_finetune = model_finetuned.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks_finetune,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)


Epoch 1/20


I0000 00:00:1779363274.857703    7791 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_405825__.409
E0000 00:00:1779363277.792038    7791 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1779363279.834211    7791 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


333/815 ━━━━━━━━━━━━━━━━━━━━ 2:10 270ms/step - accuracy: 0.7313 - loss: 2.8524

I0000 00:00:1779363394.318451    7789 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_405825__.409


815/815 ━━━━━━━━━━━━━━━━━━━━ 354s 368ms/step - accuracy: 0.7466 - loss: 2.7512 - val_accuracy: 0.7287 - val_loss: 2.8531 - learning_rate: 3.0000e-06
Epoch 2/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 195s 239ms/step - accuracy: 0.7569 - loss: 2.6245 - val_accuracy: 0.7403 - val_loss: 2.7326 - learning_rate: 3.0000e-06
Epoch 3/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 172s 211ms/step - accuracy: 0.7725 - loss: 2.5481 - val_accuracy: 0.7508 - val_loss: 2.6796 - learning_rate: 3.0000e-06
Epoch 4/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 178s 218ms/step - accuracy: 0.7860 - loss: 2.4996 - val_accuracy: 0.7520 - val_loss: 2.6426 - learning_rate: 3.0000e-06
Epoch 5/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 189s 231ms/step - accuracy: 0.7877 - loss: 2.4798 - val_accuracy: 0.7575 - val_loss: 2.6204 - learning_rate: 3.0000e-06
Epoch 6/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 265s 325ms/step - accuracy: 0.7965 - loss: 2.4484 - val_accuracy: 0.7606 - val_loss: 2.5928 - learning_rate: 3.0000e-06
Epoch 7/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 255s 313ms/

In [87]:
model = tf.keras.models.load_model("best_efficientnet_finetuned.keras")
predictions = model.predict(val_generator, steps=len(val_generator), verbose=1)
y_pred = predictions.argmax(axis=1)
y_true = val_generator.classes

val_accuracy = accuracy_score(y_true, y_pred)
print(f"Validation Accuracy: {val_accuracy:.4f}")


204/204 ━━━━━━━━━━━━━━━━━━━━ 36s 147ms/step
Validation Accuracy: 0.7962


In [85]:
test_loss, test_accuracy = model.evaluate(val_generator, verbose=1)
print(f"\nFINAL TEST ACCURACY: {test_accuracy:.2%}")

204/204 ━━━━━━━━━━━━━━━━━━━━ 33s 124ms/step - accuracy: 0.7962 - loss: 2.4747

FINAL TEST ACCURACY: 79.62%
